<a href="https://colab.research.google.com/github/Y0RLGRJALES/AUREA_AGENTE/blob/main/AUREA_AGENT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [121]:
!pip -q install pypdf
!pip -q install langchain
!pip -q install langchain-community
!pip -q install langchain-text-splitters
!pip -q install chromadb
!pip -q install sentence-transformers
!pip install -q groq


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 12.8 MB/s eta 0:00:00


In [122]:
from groq import Groq
from google.colab import userdata

client = Groq(
    api_key=userdata.get("GROQ_API_KEY")
)

print("✅ Groq conectado correctamente")

✅ Groq conectado correctamente


In [35]:
from google.colab import files

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

from transformers import AutoTokenizer
from transformers import AutoModelForCausalLM

import torch

In [4]:
uploaded = files.upload()

Saving Manual Maestro de Marca AUREA.pdf to Manual Maestro de Marca AUREA (2).pdf
Saving Política de Envíos AUREA.pdf to Política de Envíos AUREA (2).pdf
Saving Política de Devoluciones y Reembolsos AUREA.pdf to Política de Devoluciones y Reembolsos AUREA (2).pdf
Saving Manual de Pagos PSE AUREA.pdf to Manual de Pagos PSE AUREA (2).pdf
Saving Manual de Garantías AUREA.pdf to Manual de Garantías AUREA (2).pdf
Saving FAQ de Productos de Belleza y Skincare AUREA.pdf to FAQ de Productos de Belleza y Skincare AUREA (2).pdf
Saving Manual de Atención al Cliente AUREA.pdf to Manual de Atención al Cliente AUREA (2).pdf
Saving Catálogo Maestro de Productos AUREA.pdf to Catálogo Maestro de Productos AUREA (1).pdf
Saving Base de Casos Reales de Atención al Cliente AUREA.pdf to Base de Casos Reales de Atención al Cliente AUREA (1).pdf


In [5]:
documents = []

for pdf in uploaded.keys():

    loader = PyPDFLoader(pdf)

    docs = loader.load()

    for doc in docs:
        doc.metadata["source"] = pdf

    documents.extend(docs)

print("Documentos cargados:", len(documents))

Documentos cargados: 60


In [77]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 156


In [7]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

print("Embeddings cargados")

/tmp/ipykernel_9521/789477915.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Embeddings cargados


In [132]:
vectordb = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings
)

retriever = vectordb.as_retriever(
    search_type="mmr",
    search_kwargs={
        "k":8,
        "fetch_k":30,
        "lambda_mult":0.3
    }
)

In [133]:
from groq import Groq
from google.colab import userdata

groq_client = Groq(
    api_key=userdata.get("GROQ_API_KEY")
)

def generar(prompt):

    respuesta = groq_client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        temperature=0,
        max_tokens=250,
        top_p=1,
        messages=[
            {
                "role": "system",
                "content": """
Eres AUREA.

Eres el asistente virtual oficial de una tienda online de cosmeticos.

Tu unica fuente de informacion es el contexto recibido.

Nunca inventes informacion.

Nunca utilices conocimientos externos.

Nunca deduzcas informacion.

Si la respuesta no aparece claramente en el contexto responde exactamente:

No encontre informacion oficial sobre esa consulta.
"""
            },
            {
                "role": "user",
                "content": prompt
            }
        ]
    )

    return respuesta.choices[0].message.content.strip()

In [136]:
def preguntar_aurea(question):

    docs = retriever.invoke(question)

    if not docs:
        return "No encontré información oficial sobre esa consulta."

    # Eliminar duplicados
    docs_unicos = []
    vistos = set()

    for doc in docs:
        texto = doc.page_content.strip()

        if texto not in vistos:
            vistos.add(texto)
            docs_unicos.append(doc)

    docs = docs_unicos

    # Construir contexto
    contexto = "\n\n".join(
        [
            f"DOCUMENTO {i+1}\n{doc.page_content}"
            for i, doc in enumerate(docs)
        ]
    )

    prompt = f"""
CONTEXTO

{contexto}

PREGUNTA

{question}

RESPUESTA
"""

    # PRIMERO generar respuesta
    respuesta = generar(prompt)

    # MOSTRAR RESPUESTA PRIMERO
    print("\n===== RESPUESTA =====\n")
    print(respuesta)

    # DESPUÉS mostrar documentos
    print("\n===== DOCUMENTOS RECUPERADOS =====\n")

    for i, doc in enumerate(docs):

        print(f"Documento {i+1}")
        print("Fuente:", doc.metadata.get("source"))
        print(doc.page_content[:300])
        print("-" * 60)

    return respuesta

In [137]:
preguntar_aurea("¿Tienen envío gratis?")


===== RESPUESTA =====

Sí, AUREA ofrece envío gratuito para compras iguales o superiores a $150.000 COP, siempre y cuando el destino tenga cobertura logística.

===== DOCUMENTOS RECUPERADOS =====

Documento 1
Fuente: Política de Envíos AUREA (2).pdf
¿Cómo obtengo envío gratis?
Las compras iguales o superiores a $150.000 COP cuentan con envío gratuito.
¿Qué hago si mi pedido no llega?
Debes comunicarte con nuestro equipo de atención al cliente para iniciar la revisión correspondiente.
• 
• 
• 
• 
• 
• 
• 
• 
5
------------------------------------------------------------
Documento 2
Fuente: Política de Envíos AUREA (2).pdf
7. Envío Gratis
AUREA ofrece envío gratuito para compras iguales o superiores a $150.000 COP.
Condiciones
El beneficio aplica sobre el valor total de los productos adquiridos.
Aplica únicamente para destinos con cobertura logística.
No es acumulable con promociones que indiquen restricciones específ
------------------------------------------------------------
Document

'Sí, AUREA ofrece envío gratuito para compras iguales o superiores a $150.000 COP, siempre y cuando el destino tenga cobertura logística.'

In [138]:
preguntar_aurea("¿Aceptan PSE?")


===== RESPUESTA =====

Sí, AUREA acepta pagos mediante PSE (Pagos Seguros en Línea).

===== DOCUMENTOS RECUPERADOS =====

Documento 1
Fuente: Manual de Pagos PSE AUREA (2).pdf
2. ¿Qué es PSE?
PSE (Pagos Seguros en Línea) es un sistema que permite realizar pagos electrónicos directamente desde
cuentas bancarias colombianas de manera segura y rápida.
A través de PSE, los clientes pueden autorizar el pago de sus compras sin necesidad de utilizar tarjetas de
crédito.
3. Métod
------------------------------------------------------------
Documento 2
Fuente: Manual de Pagos PSE AUREA (2).pdf
MANUAL DE PAGOS PSE AUREA
Versión 1.0
Última actualización
Julio de 2026
1. Objetivo
El presente documento tiene como finalidad informar a los clientes de AUREA sobre el funcionamiento de
los pagos realizados mediante la plataforma PSE, los procedimientos de validación de transacciones y la
gestión 
------------------------------------------------------------
Documento 3
Fuente: Manual de Pagos PSE AUREA

'Sí, AUREA acepta pagos mediante PSE (Pagos Seguros en Línea).'